In [14]:
# image classification library
import cv2
import numpy as np

In [15]:
# process the image
def preprocess(image_path):
    # load image as black and white and grey
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not load {image_path}")

    # get the height and width (in pixels of image)
    h, w = img.shape

    # get the white margins for the image, take 0.03 of the height and width as the margin
    margin_y, margin_x = int(h * 0.03), int(w * 0.03)

    # crop the image on the sides
    crop = img[margin_y:h - margin_y, margin_x:w - margin_x]
    
    # isolate the actual image and clean mask
    # apply a blur to the entire image, (5, 5) is  strength of the blur, std dev of blur = 0
    blurred = cv2.GaussianBlur(crop, (5, 5), 0)

    # convert image to a binary 0 (black) or 255 (white), with pixel intensity being greater than 20 being assigned to 255
    # cv2.THRESH_BINARY is the method of thresholding, and the first output we don't really care about so _ is fine
    _, thresh = cv2.threshold(blurred, 20, 255, cv2.THRESH_BINARY)

    # only extract extreme outer contours (which is RETR_EXTERNAL) and CHAIN_APPROX_SIMPLE is compression method
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # isolate just the largest contour
    big_contour = max(contours, key=cv2.contourArea)
    
    # create a completely black (0) blank image with the same dimensions and data type as thresh
    mask = np.zeros_like(thresh, dtype=np.uint8)

    # draw the largest contour onto the blank mask in white (255)
    # 0 gets the first contour in the list, and -1 fills the inside completely
    cv2.drawContours(mask, [big_contour], 0, 255, -1)

    # create an oval-shaped kernel (MORPH_ELLIPSE) of size (15, 15) to smooth out the edges
    close_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))

    # make the foreground stronger and more present and make the background less present (MORPH_CLOSE) 
    # using the kernel (close_k) to fill in any small black holes inside the white mask
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, close_k)
    
    # make a copy of the cropped image so we don't overwrite the original
    result = crop.copy()

    # set every pixel in the copied image to black (0) wherever the mask is black (0), removing labels and background noise
    result[mask == 0] = 0

    # return both the cleaned image and the binary mask
    return result, mask

In [ ]:
# detect bright calcification specks and group them into spatial clusters
def extract_clusters(
    img_path,                 # img
    std_multiplier=5,         # higher = fewer noise specks; lower = more sensitive
    cluster_dist_ratio=0.12,  # 12% of image width merges nearby specks into larger clusters
    min_cluster_size=4        # num of points to form cluster
):

    # get img results from preprocess
    result, mask = preprocess(img_path)
    # Denoise to get rid of JPEG/film grain spikes
    # apply a median blur with a 3x3 kernel, keep sharp edges
    # kernel = num of points to look at
    # 3 x 3 means that we have our center point, and 8 points around that center
    denoised = cv2.medianBlur(result, 3)

    # scale image resolution
    # get the height (ch) and width (cw) of the denoised image in pixels
    ch, cw = denoised.shape
    # use smaller dimension out of height or width to scale
    scale_ref = min(ch, cw)
    # compute kernel size (~5% of image, minimum 25px) to erode skin edge; |1 = mod 1 = return odd number
    # odd num is necessary so we have an actual center (can't have decimal center)
    erode_size = max(25, int(scale_ref * 0.05)) | 1   # ~5% inward erosion removes skin edge
    # compute kernel size (~1.5% of image, minimum 11px) to get small points
    tophat_size = max(11, int(scale_ref * 0.015)) | 1
    # compute max distance (minimum 60px) to group points into one cluster
    link_dist = max(60, int(scale_ref * cluster_dist_ratio)) | 1

    # calculate area scaling factor relative to an 800x800 reference image (squared because area is 2D)
    scale_factor = (scale_ref / 800.0) ** 2
    # set the minimum pixel area for a valid speck (at least 3 pixels)
    min_area = max(3, int(3 * scale_factor))
    # set the maximum pixel area for a valid speck (at least 200 pixels) to filter out large artifacts
    max_area = max(200, int(200 * scale_factor))

    # White Top-Hat transform: the difference between the input image and its opening by some structuring element
    # create an oval-shaped kernel (MORPH_ELLIPSE) of size (erode_size, erode_size)
    erode_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_size, erode_size))
    # shrink the white region of the mask inward so bright outer skin edges are excluded
    interior_mask = cv2.erode(mask, erode_kernel)

    # create a small oval-shaped kernel (MORPH_ELLIPSE) of size (tophat_size, tophat_size) for speck detection
    tophat_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (tophat_size, tophat_size))
    # apply White Top-Hat transform (original minus opening) to isolate small bright spots on uneven tissue
    tophat = cv2.morphologyEx(denoised, cv2.MORPH_TOPHAT, tophat_kernel)
    # keep only top-hat pixels that fall inside the eroded interior_mask, setting everything outside to 0 (black)
    tophat = cv2.bitwise_and(tophat, tophat, mask=interior_mask)

    # Compute threshold on non-zero top-hat peaks so background zeros don't deflate std
    # extract a 1D array of all positive (> 0) pixel intensities inside the interior mask
    nonzero_pixels = tophat[(interior_mask > 0) & (tophat > 0)]
    # if no bright candidate pixels were found at all, return an empty summary and OG result image
    if len(nonzero_pixels) == 0:
        return {"total_calcifications": 0, "total_clusters": 0, "clusters": []}, result

    # calculate adaptive brightness threshold as the mean + (std_multiplier * standard deviation)
    # kind of like a CI
    thresh_val = np.mean(nonzero_pixels) + std_multiplier * np.std(nonzero_pixels)
    # convert top-hat image to binary 0 (black) or 255 (white) for pixels brighter than thresh_val;
    # _ means we dont need the first output
    # same function as before
    _, calc_mask = cv2.threshold(tophat, thresh_val, 255, cv2.THRESH_BINARY)

    # find all connected white blobs and compute their bounding-box stats and center coordinates
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        calc_mask, connectivity=8  # connectivity=8 treats diagonally touching pixels as part of the same speck
    )

    # create an empty list to store dictionaries of valid calcification specks
    calcifications = []
    # loop over every detected component ID starting at 1 (since label 0 is the black background)
    for label_id in range(1, num_labels):
        # get the total pixel count (area) of the current component from the stats matrix
        area = stats[label_id, cv2.CC_STAT_AREA]
        # keep only specks whose pixel area falls within the allowed [min_area, max_area] size range
        if min_area <= area <= max_area:
            # get the left (x) coordinate of the speck's bounding box
            x = stats[label_id, cv2.CC_STAT_LEFT]
            # get the top (y) coordinate of the speck's bounding box
            y = stats[label_id, cv2.CC_STAT_TOP]
            # get the bounding box width (bw) in pixels
            bw = stats[label_id, cv2.CC_STAT_WIDTH]
            # get the bounding box height (bh) in pixels
            bh = stats[label_id, cv2.CC_STAT_HEIGHT]

            # slice out the bounding box from the original image and extract only the pixels belonging to this label_id
            speck_pixels = result[y:y+bh, x:x+bw][labels[y:y+bh, x:x+bw] == label_id]
            # record this speck's measurements as a dictionary and append it to the calcifications list
            calcifications.append({
                "id": len(calcifications) + 1,                                               # assign a sequential 1-based ID number
                "centroid": (float(centroids[label_id][0]), float(centroids[label_id][1])), # store (x, y) center coordinates as floats
                "area_px": int(area),                                                        # store total area in pixels as an integer
                "mean_brightness": float(np.mean(speck_pixels)),                             # calculate average pixel intensity of the speck
                "max_brightness": float(np.max(speck_pixels)),                               # find peak (brightest) pixel intensity of the speck
                "integrated_brightness": float(np.sum(speck_pixels))                         # sum all pixel intensities in the speck
            })

    # merge nearby specks into clusters
    # create a completely black (0) blank image matching denoised dimensions to plot speck centers
    centroid_map = np.zeros_like(denoised, dtype=np.uint8)
    # loop through every validated calcification speck
    for c in calcifications:
        # round the floating-point (x, y) centroid coordinates to the nearest integer pixel indices
        cx, cy = int(round(c["centroid"][0])), int(round(c["centroid"][1]))
        # mark a single white (255) pixel at the speck's center location (row=cy, col=cx)
        centroid_map[cy, cx] = 255

    # create a large oval-shaped kernel of diameter link_dist to bridge gaps between nearby specks
    cluster_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (link_dist, link_dist))
    # expand (dilate function) each white center dot into a circle so nearby dots overlap and merge into blobs
    dilated = cv2.dilate(centroid_map, cluster_kernel)
    # label each merged blob with a unique cluster group ID
    _, c_labels = cv2.connectedComponents(dilated)

    # create an empty dictionary mapping each group ID to the list of specks inside it
    groups = {}
    # loop through every calcification speck again to assign it to its merged spatial group
    for c in calcifications:
        # round the speck's (x, y) centroid coordinates to integer pixel positions
        cx, cy = int(round(c["centroid"][0])), int(round(c["centroid"][1]))
        # look up which merged blob ID exists at that pixel coordinate
        grp_id = int(c_labels[cy, cx])
        # append this speck to the list for grp_id (creating a new list if grp_id isn't in groups yet)
        groups.setdefault(grp_id, []).append(c)

    # calculate padding (at least 15px, or 2.5% of scale_ref) to add around each cluster's bounding box
    pad = max(15, int(scale_ref * 0.025))
    # create an empty list to store clusters with more than min points
    clusters = []
    # iterate over each group ID and its list of member specks
    for grp_id, members in groups.items():
        # only keep groups that contain at least min_cluster_size specks
        if len(members) >= min_cluster_size:
            # collect all x-coordinates of the member specks in this cluster
            xs = [m["centroid"][0] for m in members]
            # collect all y-coordinates of the member specks in this cluster
            ys = [m["centroid"][1] for m in members]
            # compute padded left edge of the bounding box, 
            # 0 is max so we don't get a negative point
            # so it stays inside the image
            x1 = max(0, int(min(xs)) - pad)
            # compute padded top edge of the bounding box
            y1 = max(0, int(min(ys)) - pad)
            # compute padded right edge of the bounding box, clamped at image width - 1
            x2 = min(cw - 1, int(max(xs)) + pad)
            # compute padded bottom edge of the bounding box, clamped at image height - 1
            y2 = min(ch - 1, int(max(ys)) + pad)

            # save the cluster's summary metrics and member list as a dictionary
            clusters.append({
                "cluster_id": len(clusters) + 1,                                            # assign a sequential 1-based cluster ID
                "count": len(members),                                                      # record how many specks belong to this cluster
                "center": (float(np.mean(xs)), float(np.mean(ys))),                         # calculate the average (x, y) center of the cluster
                "bbox": (x1, y1, x2 - x1, y2 - y1),                                         # store bounding box as (x, y, width, height)
                "mean_brightness": float(np.mean([m["mean_brightness"] for m in members])), # average the mean brightness across all member specks
                "max_brightness": float(np.max([m["max_brightness"] for m in members])),    # find the highest max brightness among all member specks
                "members": members                                                          # include the full list of individual speck dictionaries
            })

    # Draw annotations
    # convert the 1-channel grayscale image to color image so we can draw colored annotations
    vis = cv2.cvtColor(result, cv2.COLOR_GRAY2BGR)
    # thickness of box, min thickness is 2
    line_thickness = max(2, int(scale_ref * 0.004))
    # dot radius (min 3px)
    dot_radius = max(3, int(scale_ref * 0.004))
    # text size (min 0.5)
    font_scale = max(0.5, scale_ref / 1200.0)

    # loop through every detected calcification speck to draw it on the visualization image
    for c in calcifications:
        # round the speck's (x, y) centroid to integer pixel coordinates
        cx, cy = int(round(c["centroid"][0])), int(round(c["centroid"][1]))
        # draw a filled (-1) yellow circle (BGR: 0, 255, 255) at each speck's centroid
        cv2.circle(vis, (cx, cy), dot_radius, (0, 255, 255), -1)

    # loop through every cluster to draw its bounding box and text label
    for cl in clusters:
        # unpack the cluster's box into top-left (x, y) and dimensions (bw, bh)
        x, y, bw, bh = cl["bbox"]
        # draw a red rectangle (BGR: 0, 0, 255) around the cluster using top-left and bottom-right corners
        cv2.rectangle(vis, (x, y), (x + bw, y + bh), (0, 0, 255), line_thickness)
        # draw the cluster ID, speck count, and max brightness label just above the top-left corner of the box
        cv2.putText(
            vis,                                                                      # target color image to draw text onto
            f"C{cl['cluster_id']} (n={cl['count']}, max={cl['max_brightness']:.0f})", # formatted label string (e.g., "C1 (n=6, max=215)")
            (x, max(y - 10, 25)),                                                     # text bottom-left origin, clamped at y=25 so it doesn't clip at the top
            cv2.FONT_HERSHEY_SIMPLEX,                                                 # standard sans-serif OpenCV font style
            font_scale,                                                               # scaled font size
            (0, 0, 255),                                                              # red text color
            line_thickness,                                                           # stroke thickness of the text characters
        )

    # overall statistics, point count, and cluster details
    summary = {
        "total_calcifications": len(calcifications),                                                                          # total number of individual specks detected
        "total_clusters": len(clusters),                                                                                      # total number of multi-speck clusters found
        "overall_mean_brightness": float(np.mean([c["mean_brightness"] for c in calcifications])) if calcifications else 0.0, # average brightness of all specks (or 0.0 if none)
        "calcifications": calcifications,                                                                                     # list of all individual calcification dictionaries
        "clusters": clusters                                                                                                  # list of all qualifying cluster dictionaries
    }
    # return both the summary and img w annotations
    return summary, vis

In [17]:
features, annotated_img = extract_clusters("1-231.jpg")

print(f"Total specks: {features['total_calcifications']} | Total clusters: {features['total_clusters']}")
for cl in features["clusters"]:
    print(f"Cluster {cl['cluster_id']}: count={cl['count']}, mean_brightness={cl['mean_brightness']:.1f}, max_brightness={cl['max_brightness']:.1f}")

cv2.namedWindow('features', cv2.WINDOW_NORMAL)
cv2.resizeWindow('features', 500, 700)
cv2.imshow('features', annotated_img)

cv2.waitKey(0)
cv2.destroyAllWindows()

Total specks: 66 | Total clusters: 5
Cluster 1: count=17, mean_brightness=160.9, max_brightness=187.0
Cluster 2: count=9, mean_brightness=162.1, max_brightness=198.0
Cluster 3: count=7, mean_brightness=162.2, max_brightness=192.0
Cluster 4: count=14, mean_brightness=157.5, max_brightness=176.0
Cluster 5: count=4, mean_brightness=139.3, max_brightness=169.0


In [19]:
features, annotated_img = extract_clusters("1-263.jpg")

# print(f"Total specks: {features['total_calcifications']} | Total clusters: {features['total_clusters']}")
# for cl in features["clusters"]:
#     print(f"Cluster {cl['cluster_id']}: count={cl['count']}, mean_brightness={cl['mean_brightness']:.1f}, max_brightness={cl['max_brightness']:.1f}")
height, width = annotated_img.shape[:2]

cv2.namedWindow('features', cv2.WINDOW_NORMAL)
cv2.resizeWindow('features', min(700, height * 3) , min(700, width * 3))
cv2.imshow('features', annotated_img)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [20]:
features, annotated_img = extract_clusters("1-108.jpg")

# print(f"Total specks: {features['total_calcifications']} | Total clusters: {features['total_clusters']}")
# for cl in features["clusters"]:
#     print(f"Cluster {cl['cluster_id']}: count={cl['count']}, mean_brightness={cl['mean_brightness']:.1f}, max_brightness={cl['max_brightness']:.1f}")

height, width = annotated_img.shape[:2]

cv2.namedWindow('features', cv2.WINDOW_NORMAL)
cv2.resizeWindow('features', min(700, height * 3) , min(700, width * 3))
cv2.imshow('features', annotated_img)

cv2.waitKey(0)
cv2.destroyAllWindows()

In [21]:
# should be benign
features, annotated_img = extract_clusters("1-193.jpg")

print(f"Total specks: {features['total_calcifications']} | Total clusters: {features['total_clusters']}")
for cl in features["clusters"]:
    print(f"Cluster {cl['cluster_id']}: count={cl['count']}, mean_brightness={cl['mean_brightness']:.1f}, max_brightness={cl['max_brightness']:.1f}")

cv2.namedWindow('features', cv2.WINDOW_NORMAL)
cv2.resizeWindow('features', 500, 700)
cv2.imshow('features', annotated_img)

cv2.waitKey(0)
cv2.destroyAllWindows()

Total specks: 10 | Total clusters: 0
